# Experiment: Case-001 Mitapivat Clinician Answer Action Ladder

Objective:
- Validate that a clinician answer changes only the research queue.
- Confirm each allowed input label maps to a private action, public action, and next state.

Success criteria:
- six input labels are present;
- six ladder rows are present;
- public actions stay label-only and avoid treatment, access, or eligibility language.


In [ ]:
# Setup: imports and source path
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "data/regulatory/mitapivat/2026-05-18-mitapivat-clinician-answer-action-ladder.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

In [ ]:
required_labels = {
    "review_not_ready_missing_records",
    "not_clinically_relevant_now",
    "review_with_hematologist_only",
    "refer_to_trial_or_access_owner_after_clinician_review",
    "safety_review_required_before_any_discussion",
    "lane_paused_until_pediatric_results_or_recruitment",
}

unsafe_public_terms = {
    " start ",
    " stop ",
    " dose",
    " dosing",
    " recommend",
    " eligible",
    " eligibility",
    " import",
    " travel",
}


def has_unsafe_term(text: str) -> bool:
    """Return True when public-facing text asks for unsafe action."""
    normalized = f" {text.lower()} "
    return any(term in normalized for term in unsafe_public_terms)


ladder = gate["action_ladder"]
assert gate["gate_label"] == "mitapivat_clinician_answer_action_ladder_ready"
assert set(gate["allowed_input_labels"]) == required_labels
assert {item["input_label"] for item in ladder} == required_labels
assert len(gate["source_checks"]) >= 5

for item in ladder:
    assert item["private_action"]
    assert item["public_repo_action"]
    assert item["next_state"]
    assert not has_unsafe_term(item["public_repo_action"])

decision = {
    "gate_label": gate["gate_label"],
    "input_labels_checked": len(gate["allowed_input_labels"]),
    "ladder_rows_checked": len(ladder),
    "public_action": "record_labels_only_no_treatment_decision",
}
decision